In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import pandas as pd
from jax import grad
from tqdm import tqdm

from trunx.gp3.PG3_model_impl import prepare_data
from trunx.gp3.run_3pg import run_3pg

In [ ]:
file_path = "./data/solling_data.xlsx"
initial_state, climate, params, site_data, species_data, n_species, species_names = prepare_data(
    file_path
)

In [ ]:
# Observed data
try:
    observed_data = pd.read_excel(file_path, sheet_name="observed")
    print(observed_data.head())
    obs_times = jnp.array(observed_data["idx"].values, dtype=jnp.int32)
    obs_values = jnp.array(observed_data["DBH"].values, dtype=jnp.float32)
    print(f"\nObservation times (months): {obs_times}")
    print(f"Observed DBH values (cm): {obs_values}")
except Exception:
    print("No observed data found. Creating synthetic observations for demonstration.")
    _, outputs = run_3pg(initial_state, climate, params, site_data, species_data, n_species)
    obs_indices = [12, 24, 36, 48, 60, 72, 84, 96, 108, 120]
    obs_indices = [i for i in obs_indices if i < len(outputs["DBH"])]
    obs_times = jnp.array(obs_indices, dtype=jnp.int32)
    obs_values = outputs["DBH"][obs_times].reshape(-1) + jnp.array(
        np.random.normal(0, 0.5, len(obs_indices))
    )

    print(f"Synthetic observations at months: {obs_times}")
    print(f"Observed DBH values (cm): {obs_values}")

In [ ]:
def transform_parameters(params_dict):
    """
    Transform parameters for unconstrained optimization.

    - Positive parameters: log transform
    - Parameters in (0,1): logit transform
    """
    transformed = []
    names = []

    for name, value in params_dict.items():
        names.append(name)
        if name == "Y":  # Y is between 0 and 1
            # Logit transform: log(p/(1-p))
            logit_val = jnp.log(value / (1 - value + 1e-8))
            transformed.append(logit_val)
            print(f"{name}: {value:.4f} -> logit({value:.4f}) = {logit_val:.4f}")
        else:  # Positive parameters
            log_val = jnp.log(value)
            transformed.append(log_val)
            print(f"{name}: {value:.4f} -> log({value:.4f}) = {log_val:.4f}")

    return jnp.array(transformed), names


def inverse_transform(transformed_vec, names):
    """Transform back to original parameter space."""
    params_dict = {}
    for i, name in enumerate(names):
        if name == "Y":
            params_dict[name] = float(jax.nn.sigmoid(transformed_vec[i]))
        else:
            params_dict[name] = float(jnp.exp(transformed_vec[i]))
    return params_dict

In [ ]:
# example_params = {
#     'alphaCx': 0.05,
#     'CoeffCond': 0.05,
#     'Y': 0.47
# }

# transformed_vec, names = transform_parameters(example_params)
# print(f"Transformed vector: {transformed_vec}")

# # Transform back
# original = inverse_transform(transformed_vec, names)
# print(f"Original parameters: {original}")

In [ ]:
def create_loss_function(
    initial_state,
    climate,
    base_params,
    site_data,
    species_data,
    n_species,
    obs_times,
    obs_values,
    species_idx=0,
):
    """
    Create a loss function for a specific species.

    Parameters
    ----------
    species_idx : int
        Index of the species to optimize for
    """

    def loss_fn(transformed_params, param_names):
        updated_params = base_params
        for i, name in enumerate(param_names):
            if name == "Y":
                val = jax.nn.sigmoid(transformed_params[i])
            else:
                val = jnp.exp(transformed_params[i])
            updated_params = updated_params._replace(**{name: val})

        _, outputs = run_3pg(
            initial_state, climate, updated_params, site_data, species_data, n_species
        )

        pred_dbh = outputs["DBH"][obs_times, species_idx]

        mse = jnp.mean((pred_dbh - obs_values) ** 2)

        # Regularization
        # l2_reg = 0.001 * jnp.sum(transformed_params**2)

        return mse  # + l2_reg

    return loss_fn

In [ ]:
param_names = ["alphaCx", "CoeffCond", "Y"]

initial_params_dict = {
    "alphaCx": params.alphaCx[0],
    "CoeffCond": params.CoeffCond[0],
    "Y": params.Y[0],
}

# Transform initial parameters
transformed_init, _ = transform_parameters(initial_params_dict)

names = []
transformed = []
for name, value in initial_params_dict.items():
    names.append(name)
    transformed.append(value)

params_init = jnp.array(transformed)

# Create loss function
loss_fn = create_loss_function(
    initial_state,
    climate,
    params,
    site_data,
    species_data,
    n_species,
    obs_times,
    obs_values,
    species_idx=0,
)

In [ ]:
initial_loss = loss_fn(transformed_init, param_names)
print(f"Initial loss (MSE): {initial_loss:.6f}")
print(f"Root MSE: {np.sqrt(initial_loss):.4f} cm")

In [ ]:
def gradient_descent_optimization(
    loss_fn, initial_params, param_names, learning_rate=0.01, n_iterations=500, verbose=True
):
    """Perform gradient descent optimization.

    Returns
    -------
    dict with optimal paramters, loss history and parameter history
    """
    # optimizer = optax.adam(learning_rate)
    # optimizer = optax.adam(learning_rate, b1=0.5, b2=0.99)
    optimizer = optax.adamw(learning_rate, weight_decay=0.001, b1=0.5)
    opt_state = optimizer.init(initial_params)

    grad_fn = grad(loss_fn, argnums=0)

    loss_history = []
    param_history = []
    params_current = initial_params.copy()

    iterator = (
        tqdm(range(n_iterations), desc="Optimizing parameters") if verbose else range(n_iterations)
    )

    best_loss = float("inf")
    best_params = params_current.copy()
    patience_counter = 0
    early_stopping_patience = 50

    for i in iterator:
        loss_value = loss_fn(params_current, param_names)
        grads = grad_fn(params_current, param_names)

        # params_current = params_current - learning_rate * grads
        updates, opt_state = optimizer.update(grads, opt_state, params_current)
        params_current = optax.apply_updates(params_current, updates)

        loss_history.append(float(loss_value))
        param_history.append(jnp.asarray(params_current))

        if loss_value - best_loss < 1e-5:
            best_loss = loss_value
            best_params = jnp.asarray(params_current)
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= early_stopping_patience:
            print(f"Early stopping at iteration {i} with best loss {best_loss:.6f}")
            break

        if verbose and (i % 10 == 0 or i == n_iterations - 1):
            print(f"Iteration {i}, Loss: {loss_value:.6f}")

    optimal_params = inverse_transform(best_params, param_names)

    return {
        "optimal_params": optimal_params,
        "loss_history": loss_history,
        "params_history": param_history,
        "best_loss": best_loss,
        "n_iterations": len(loss_history),
    }

In [ ]:
print("Starting gradient descent optimization...")
result = gradient_descent_optimization(
    loss_fn=loss_fn,
    initial_params=transformed_init,
    param_names=param_names,
    learning_rate=0.01,
    n_iterations=100,
    verbose=True,
)

print("\n=== Optimization Results ===")
print(f"Optimal parameters: {result['optimal_params']}")
print(f"Final loss: {result['best_loss']:.6f}")
print(f"Final RMSE: {np.sqrt(result['best_loss']):.4f} cm")
print(f"Converged in {result['n_iterations']} iterations")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Loss curve
axes[0, 0].plot(result["loss_history"])
axes[0, 0].set_xlabel("Iteration")
axes[0, 0].set_ylabel("Loss (MSE)")
axes[0, 0].set_title("Loss History")
axes[0, 0].set_yscale("log")
axes[0, 0].grid(True)

# RMSE curve
rmse_history = np.sqrt(result["loss_history"])
axes[0, 1].plot(rmse_history)
axes[0, 1].set_xlabel("Iteration")
axes[0, 1].set_ylabel("RMSE (cm)")
axes[0, 1].set_title("Root Mean Square Error")
axes[0, 1].grid(True)

# Parameter trajectories (in original space)
params_history_original = []
for p_vec in result["params_history"]:
    p_dict = inverse_transform(p_vec, param_names)
    params_history_original.append(p_dict)

for name in param_names:
    history = [p[name] for p in params_history_original]
    axes[1, 0].plot(history, label=name)
axes[1, 0].set_xlabel("Iteration")
axes[1, 0].set_ylabel("Parameter Value")
axes[1, 0].set_title("Parameter Trajectories")
axes[1, 0].legend()
axes[1, 0].grid(True)

# Parameter convergence (relative change)
for name in param_names:
    history = [p[name] for p in params_history_original]
    if len(history) > 1:
        rel_change = np.abs(np.diff(history) / (np.array(history[:-1]) + 1e-8))
        axes[1, 1].plot(rel_change, label=name)
axes[1, 1].set_xlabel("Iteration")
axes[1, 1].set_ylabel("Relative Change")
axes[1, 1].set_title("Parameter Convergence Rate")
axes[1, 1].set_yscale("log")
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
def run_model_with_params(parameter_dict):
    """Run model with given parameters."""
    updated_params = params
    for name, value in parameter_dict.items():
        if hasattr(updated_params, name):
            current_val = getattr(updated_params, name)
            if isinstance(current_val, jnp.ndarray) and len(current_val.shape) > 0:
                value = jnp.full_like(current_val, value, dtype=jnp.float64)
            updated_params = updated_params._replace(**{name: value})

    _, outputs = run_3pg(
        initial_state, climate, updated_params, site_data, species_data, n_species
    )
    return outputs


initial_outputs = run_model_with_params(initial_params_dict)

optimal_outputs = run_model_with_params(result["optimal_params"])


time_months = np.arange(len(initial_outputs["DBH"]))
time_years = time_months / 12

for idx, species in enumerate(species_data.specie[:1]):
    plt.plot(
        time_years, initial_outputs["DBH"][:, idx], "--", label=f"{species} (initial)", alpha=0.7
    )
    plt.plot(
        time_years, optimal_outputs["DBH"][:, idx], "-", label=f"{species} (optimized)", alpha=0.7
    )

obs_years = np.array(obs_times) / 12
plt.scatter(obs_years, obs_values, color="red", s=50, label="Observations", zorder=5)

plt.xlabel("Time")
plt.ylabel("DBH (cm)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()